# Grupo 6 – Base Adult – KNN, SVM e MLP
**Inteligência de Negócios – BSI – 2026-2**

Notebook do grupo. Seções da **Pessoa 1**: análise exploratória (3.1), pré-processamento e validação (3.2), KNN e SVM.
A **Pessoa 2** adiciona o MLP usando exatamente o mesmo `prep`, `skf` e `avaliar` (seção reservada mais abaixo).

> Para os tempos serem comparáveis entre modelos, **o notebook inteiro deve ser executado no mesmo ambiente (Google Colab)**.

## A1. Ambiente

In [ ]:
# No Colab, só falta o ucimlrepo (pandas, sklearn, seaborn etc. já vêm instalados)
%pip install -q ucimlrepo

In [ ]:
import os, json, time, platform
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import sklearn

os.makedirs('figuras', exist_ok=True)
os.makedirs('resultados', exist_ok=True)
SEED = 42
print('Python', platform.python_version(), '| sklearn', sklearn.__version__,
      '| pandas', pd.__version__, '| CPUs', os.cpu_count())

## A2. Análise exploratória (requisito 3.1)
### Carregamento
A base é baixada do UCI (id=2: 32.561 registros de treino + 16.281 de teste, unidos). Uma cópia local
(`adult_raw.csv`) é salva para que o notebook não dependa do servidor do UCI.

In [ ]:
if os.path.exists('adult_raw.csv'):
    df = pd.read_csv('adult_raw.csv')
else:
    from ucimlrepo import fetch_ucirepo
    adult = fetch_ucirepo(id=2)
    df = pd.concat([adult.data.features, adult.data.targets], axis=1)
    df.to_csv('adult_raw.csv', index=False)

print('Ausentes já como NaN no arquivo original:')
print(df.isna().sum()[lambda s: s > 0])

### Limpeza de marcações
- Espaços extras nos textos são removidos.
- O rótulo vem em duas grafias (`<=50K` no treino e `<=50K.` no teste) – unificamos.
- A base marca ausentes **de duas formas**: `NaN` e `'?'`. Convertemos `'?'` para `NaN`.

In [ ]:
for c in df.select_dtypes(exclude='number').columns:
    df[c] = df[c].str.strip()
df['income'] = df['income'].str.rstrip('.')   # '<=50K.' e '<=50K' viram a mesma classe
df = df.replace('?', np.nan)                 # a base usa '?' para ausente

### 1) Registros, atributos e tipos

In [ ]:
print('Registros x colunas:', df.shape)
print(df.dtypes)

### 2) Variável-alvo
`income`: `<=50K` ou `>50K` (renda anual). Classificação binária; a classe positiva (1) será `>50K`.

### 3) Ausentes, duplicados e inconsistências

In [ ]:
aus = df.isna().sum()
print(pd.DataFrame({'ausentes': aus, '%': (100 * aus / len(df)).round(2)})[aus > 0])
print('\nRegistros com pelo menos um ausente:', df.isna().any(axis=1).sum())
print('Duplicados exatos:', df.duplicated().sum())

In [ ]:
df.describe().T

In [ ]:
# capital-gain = 99999 é o teto de registro do censo (valor truncado), não erro de digitação.
print('Registros com capital-gain = 99999:', (df['capital-gain'] == 99999).sum())
print('% de zeros em capital-gain:', round(100 * (df['capital-gain'] == 0).mean(), 1))
print('% de zeros em capital-loss:', round(100 * (df['capital-loss'] == 0).mean(), 1))
# education e education-num: relação 1 para 1?
print('Valores de education-num por categoria de education (máx):',
      df.groupby('education')['education-num'].nunique().max())

### 4) Distribuição da variável-alvo

In [ ]:
print(df['income'].value_counts())
print(df['income'].value_counts(normalize=True).round(4))
ax = df['income'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], rot=0)
ax.set_title('Distribuição de income'); ax.set_ylabel('Registros')
plt.tight_layout(); plt.savefig('figuras/dist_income.png', dpi=300); plt.show()

### Figuras complementares

In [ ]:
# Histogramas de age, hours-per-week e education-num
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, c in zip(axs, ['age', 'hours-per-week', 'education-num']):
    sns.histplot(data=df, x=c, hue='income', bins=30, ax=ax, multiple='stack')
    ax.set_title(c)
plt.tight_layout(); plt.savefig('figuras/histogramas.png', dpi=300); plt.show()

In [ ]:
# Proporção de >50K em algumas categóricas
cats = ['sex', 'relationship', 'marital-status', 'workclass']
fig, axs = plt.subplots(2, 2, figsize=(14, 9))
for ax, c in zip(axs.ravel(), cats):
    prop = pd.crosstab(df[c], df['income'], normalize='index').sort_values('>50K')
    prop.plot(kind='barh', stacked=True, ax=ax, color=['#4C72B0', '#DD8452'], legend=False)
    ax.set_title(f'income por {c}'); ax.set_xlabel('proporção'); ax.set_ylabel('')
axs[0, 0].legend(title='income', loc='lower right')
plt.tight_layout(); plt.savefig('figuras/categoricas_por_income.png', dpi=300); plt.show()

In [ ]:
# Boxplot das numéricas por income
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, c in zip(axs, ['age', 'hours-per-week', 'education-num']):
    sns.boxplot(data=df, x='income', y=c, ax=ax)
plt.tight_layout(); plt.savefig('figuras/boxplots.png', dpi=300); plt.show()

In [ ]:
# Matriz de correlação das variáveis numéricas
num = df.select_dtypes(include='number')
plt.figure(figsize=(7, 5.5))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlação (Pearson) das variáveis numéricas')
plt.tight_layout(); plt.savefig('figuras/correlacao.png', dpi=300); plt.show()

### Conclusão da EDA
- **48.842 registros e 15 colunas** (6 numéricas, 8 categóricas e o alvo).
- **Desbalanceamento:** ~76,1% `<=50K` e ~23,9% `>50K`. Um classificador que sempre prevê `<=50K` já teria
  ~76% de acurácia, por isso precisão, revocação e F1 da classe `>50K` são essenciais na interpretação.
- **Ausentes** apenas em `workclass` (2.799), `occupation` (2.809) e `native-country` (857).
- **52 registros duplicados exatos.**
- `capital-gain` e `capital-loss` são muito assimétricas (≈92% e ≈95% de zeros); 244 registros têm
  `capital-gain = 99999`, que é o teto do censo – mantidos, pois representam rendas reais muito altas.
- `education` e `education-num` codificam a mesma informação (relação 1 para 1).
- `fnlwgt` tem correlação praticamente nula com as demais numéricas – é um peso amostral do censo.

## A3. Pré-processamento e validação (requisito 3.2)

| Decisão | Justificativa |
|---|---|
| Remover duplicados **antes** de remover `fnlwgt` | Sem `fnlwgt`, pessoas diferentes ficariam parecendo iguais. Só os duplicados exatos saem. |
| Remover `fnlwgt` | Peso amostral do censo, não uma característica da pessoa. |
| Remover `education` | Redundante com `education-num`. |
| Imputar ausentes: moda (categóricas) e mediana (numéricas) | Mantém todos os registros. |
| One-hot nas categóricas | KNN, SVM e MLP não lidam com texto. |
| StandardScaler nas numéricas | KNN e SVM dependem de distância; o MLP converge melhor com dados padronizados. |
| Tudo dentro de um `Pipeline` | Imputador, scaler e encoder são ajustados **só no treino de cada fold** (sem vazamento de dados). |
| `StratifiedKFold(5, shuffle=True, random_state=42)` | **Mesma divisão para todos os modelos**, mantendo a proporção das classes em cada fold. |

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = df.drop_duplicates()                      # 1º: duplicados exatos
df = df.drop(columns=['fnlwgt', 'education'])  # 2º: colunas inapropriadas/redundantes

y = (df['income'] == '>50K').astype(int)       # 1 = classe positiva (>50K)
X = df.drop(columns='income').reset_index(drop=True)
y = y.reset_index(drop=True)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])

# MESMA divisão para TODOS os modelos
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Após limpeza:', X.shape, '| numéricas:', num_cols)
print('Categóricas:', cat_cols)
print('Proporção >50K:', round(y.mean(), 4))
print('Atributos após one-hot (ajustado na base toda só para contar):',
      prep.fit_transform(X).shape[1])

In [ ]:
# Registro dos folds (prova de que todos os modelos usam a mesma divisão)
fold_id = np.empty(len(X), dtype=int)
for i, (tr, te) in enumerate(skf.split(X, y)):
    fold_id[te] = i
    print(f'Fold {i}: treino={len(tr)}  teste={len(te)}  %>50K no teste={y.iloc[te].mean():.4f}')
pd.DataFrame({'fold': fold_id}).to_csv('resultados/folds.csv', index_label='linha')

## A4. Função de avaliação (compartilhada com a Pessoa 2)
Um único treino por fold; coleta acurácia, precisão, revocação, F1 (da classe positiva `>50K`) e a matriz
de confusão somada dos 5 folds. O desvio padrão é o populacional (`np.std`, ddof=0) entre os 5 folds.

`max_treino` (opcional, padrão `None` = usa tudo): **somente** como último recurso para o SVM RBF, reduz o
**treino** de cada fold a uma amostra estratificada; o **teste continua sendo o fold completo**, então os folds
continuam os mesmos dos demais modelos. Se for usado, tem que ser declarado no artigo.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

resultados = []   # lista de dicionários (vira tabela)
matrizes = {}     # nome -> matriz de confusão somada dos 5 folds

def avaliar(nome, clf, max_treino=None):
    pipe = Pipeline([('prep', prep), ('clf', clone(clf))])
    cm_total = np.zeros((2, 2), dtype=int)
    metr = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
    t0 = time.time()
    for tr, te in skf.split(X, y):
        if max_treino is not None and len(tr) > max_treino:
            tr, _ = train_test_split(tr, train_size=max_treino, stratify=y.iloc[tr],
                                     random_state=SEED)
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[te])
        metr['acc'].append(accuracy_score(y.iloc[te], pred))
        metr['prec'].append(precision_score(y.iloc[te], pred, zero_division=0))
        metr['rec'].append(recall_score(y.iloc[te], pred))
        metr['f1'].append(f1_score(y.iloc[te], pred))
        cm_total += confusion_matrix(y.iloc[te], pred)
    linha = {'modelo': nome, 'tempo_s': round(time.time() - t0, 1),
             'treino_por_fold': max_treino or 'completo'}
    for k, v in metr.items():
        linha[k + '_media'] = np.mean(v)
        linha[k + '_dp'] = np.std(v)
    resultados.append(linha)
    matrizes[nome] = cm_total
    print(f"{nome}: acc={linha['acc_media']:.4f} prec={linha['prec_media']:.4f} "
          f"rec={linha['rec_media']:.4f} f1={linha['f1_media']:.4f} ({linha['tempo_s']}s)")

## A5. KNN
Varia o número de vizinhos `k` (5, 15, 31) com voto uniforme e compara com o voto ponderado pela distância
(k=15). k pequeno tende a overfitting (fronteira ruidosa); k grande suaviza demais.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [5, 15, 31]:
    avaliar(f"KNN k={k} uniform", KNeighborsClassifier(n_neighbors=k, weights='uniform', n_jobs=-1))
avaliar("KNN k=15 distance", KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1))

**Interpretação (KNN).** O F1 da classe `>50K` sobe de k=5 para k=15 e praticamente estabiliza em k=31
(diferença de ~0,001, bem menor que o desvio padrão entre folds, ~0,006). Com k=5 a fronteira é ruidosa
(overfitting): precisão menor e mais falsos positivos. O voto ponderado pela distância (k=15) ficou **abaixo** do
voto uniforme com o mesmo k, porque dá peso excessivo aos vizinhos mais próximos e se comporta como um k menor.
**Melhor configuração: KNN k=31 uniform** (k=15 uniform é praticamente equivalente).

## A6. SVM
Kernel RBF com C=1 e C=10 (C maior = margem mais rígida, menos regularização) e um SVM linear como
comparação rápida. O SVM RBF é a etapa mais lenta do trabalho (vários minutos por fold).
`cache_size` maior só acelera o treino, não muda o resultado.

In [ ]:
from sklearn.svm import SVC, LinearSVC

avaliar("SVM RBF C=1",    SVC(kernel='rbf', C=1,  gamma='scale', cache_size=2000))
avaliar("SVM RBF C=10",   SVC(kernel='rbf', C=10, gamma='scale', cache_size=2000))
avaliar("SVM Linear C=1", LinearSVC(C=1, max_iter=5000))   # rápido, útil como comparação

**Interpretação (SVM).** O RBF com C=1 teve a maior acurácia e precisão; com C=10 (menos regularização) a
revocação e o F1 sobem um pouco, mas as diferenças estão dentro do desvio padrão entre folds, e o treino fica
mais lento. O SVM linear chega perto do RBF (F1 ~0,655 contra ~0,666) com um custo computacional centenas de
vezes menor, o que indica que, após o one-hot, as classes são quase linearmente separáveis.
**Melhor configuração pelo F1: SVM RBF C=10** (C=1 se o critério for acurácia).

Em todos os modelos a revocação da classe `>50K` fica em ~0,59–0,61: cerca de 40% das pessoas de renda alta são
classificadas como `<=50K`. Os erros são dominados por **falsos negativos**, efeito do desbalanceamento (76/24).

## B. Pessoa 2: MLP (Multilayer Perceptron) e comparação de configurações
Figuras e tabelas desta parte são salvas na pasta `Pessoa 2/`.
Usa **exatamente** o mesmo `prep`, `skf` e `avaliar` da Pessoa 1 (nada é redefinido aqui), então o MLP é
avaliado nos mesmos 5 folds do KNN e do SVM.

### B1. Parâmetros do `MLPClassifier` comparados
| Parâmetro | O que controla |
|---|---|
| `hidden_layer_sizes` | Arquitetura: nº de camadas ocultas e de neurônios em cada uma, ex.: `(128, 64)` = 2 camadas. |
| `activation` | Função de ativação das camadas ocultas: `relu` (max(0, x)) ou `tanh`. |
| `alpha` | Força da regularização L2 (penaliza pesos grandes, reduz overfitting). |
| `learning_rate_init` | Tamanho do passo do otimizador Adam. Grande demais oscila, pequeno demais demora. |
| `early_stopping` | Separa 10% do **treino** de cada fold como validação interna e para quando a acurácia nela não melhora por `n_iter_no_change`=10 épocas (máx. 300). O fold de teste nunca é usado nisso. |

### B3. Experimentos do MLP (requisito 3.4.3 do Grupo 6)
Para a comparação ser controlada, há uma **configuração base** (1 camada de 64 neurônios, `relu`,
`alpha=1e-4`, `lr=1e-3`) e cada experimento muda **um único aspecto** em relação a ela:

| Configuração | Dimensão variada | O que muda em relação à base |
|---|---|---|
| MLP (64) relu | (base) | – |
| MLP (128,64) relu | Arquitetura | 2 camadas ocultas (128 e 64 neurônios) |
| MLP (128,64,32) relu | Arquitetura | 3 camadas ocultas (128, 64 e 32 neurônios) |
| MLP (64) tanh | Ativação | `tanh` no lugar de `relu` |
| MLP (64) alpha=1e-2 | Regularização | L2 100 vezes mais forte |
| MLP (64) lr=1e-2 | Taxa de aprendizado | passo inicial 10 vezes maior |

Otimizador Adam, `random_state=42` (mesma inicialização dos pesos em todas as execuções).

In [ ]:
from sklearn.neural_network import MLPClassifier

def mlp(hidden=(64,), act='relu', alpha=1e-4, lr=1e-3):
    # Configuração base; cada experimento altera UM parâmetro em relação a ela
    return MLPClassifier(hidden_layer_sizes=hidden, activation=act, alpha=alpha,
                         learning_rate_init=lr, max_iter=300, early_stopping=True,
                         n_iter_no_change=10, random_state=SEED)

configs_mlp = {
    "MLP (64) relu":        mlp(),                       # base
    "MLP (128,64) relu":    mlp(hidden=(128, 64)),       # arquitetura
    "MLP (128,64,32) relu": mlp(hidden=(128, 64, 32)),   # arquitetura
    "MLP (64) tanh":        mlp(act='tanh'),             # ativação
    "MLP (64) alpha=1e-2":  mlp(alpha=1e-2),             # regularização
    "MLP (64) lr=1e-2":     mlp(lr=1e-2),                # taxa de aprendizado
}
for nome, clf in configs_mlp.items():
    avaliar(nome, clf)

### B4. Curvas de aprendizado (ilustrativas)
Cada configuração é treinada **uma vez na base inteira** só para mostrar como o treino evolui (perda de treino e
acurácia na validação interna do early stopping) e quantas épocas foram necessárias.
**Essas curvas não são resultados de desempenho:** as métricas oficiais vêm da validação cruzada (seção B3).

In [ ]:
os.makedirs('Pessoa 2/figuras', exist_ok=True); os.makedirs('Pessoa 2/resultados', exist_ok=True)
Xt = clone(prep).fit_transform(X)        # só para as curvas ilustrativas
modelos_mlp = {}
for nome, clf in configs_mlp.items():
    m = clone(clf).fit(Xt, y)
    modelos_mlp[nome] = m
    print(f"{nome:22s} épocas={m.n_iter_:3d}  perda final={m.loss_curve_[-1]:.4f}  "
          f"melhor acurácia na validação interna={m.best_validation_score_:.4f}")

# Melhor MLP pelo F1 médio da validação cruzada
res_mlp = pd.DataFrame(resultados).query("modelo.str.startswith('MLP')", engine='python')
melhor_mlp = res_mlp.loc[res_mlp['f1_media'].idxmax(), 'modelo']
print('\nMelhor MLP (F1 na validação cruzada):', melhor_mlp)

cores = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for cor, (nome, m) in zip(cores, modelos_mlp.items()):
    axs[0].plot(m.loss_curve_, color=cor, lw=2, label=nome)
    axs[1].plot(m.validation_scores_, color=cor, lw=2, label=nome)
axs[0].set(title='Perda (log-loss) no treino', xlabel='Época', ylabel='Loss')
axs[1].set(title='Acurácia na validação interna (early stopping)', xlabel='Época', ylabel='Acurácia')
for ax in axs:
    ax.grid(alpha=.3); ax.spines[['top', 'right']].set_visible(False)
axs[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig('Pessoa 2/figuras/curvas_mlp.png', dpi=300); plt.show()

# Curva de perda só do melhor MLP (figura menor, para o artigo)
m = modelos_mlp[melhor_mlp]
plt.figure(figsize=(5, 3.2))
plt.plot(m.loss_curve_, color='#2a78d6', lw=2)
plt.xlabel('Época'); plt.ylabel('Loss'); plt.title(f'Curva de perda do {melhor_mlp}')
plt.grid(alpha=.3); plt.gca().spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('Pessoa 2/figuras/loss_mlp.png', dpi=300); plt.show()

**Interpretação (MLP).** Valores: F1 da classe `>50K`, média ± desvio padrão dos 5 folds.
- **As 5 primeiras configurações empatam na prática:** F1 entre 0,6718 e 0,6762, uma diferença menor que o desvio
  padrão entre folds (~0,012). Mudar arquitetura, ativação ou regularização quase não alterou o desempenho.
- **Arquitetura:** a rede de 2 camadas **(128,64) teve o melhor F1 (0,6762) e a melhor revocação (0,6243)**, mas
  ganhou só +0,004 de F1 sobre a base, com ~1,6× o tempo. A rede de 3 camadas **(128,64,32) não melhorou nada**
  (F1 0,6718) e foi a mais lenta (~1,9× a base). Nas curvas, ela atinge a **menor perda de treino** (0,238) e a
  **pior acurácia na validação interna** (~0,857): sinal de **overfitting** (capacidade a mais decora o treino).
- **Ativação:** `tanh` teve o mesmo F1 da `relu` (0,6723), porém foi mais lenta (19,0 s contra 13,2 s).
- **Regularização:** `alpha=1e-2` também empatou (0,6722). O early stopping já regulariza a rede, então
  aumentar o L2 não trouxe ganho.
- **Taxa de aprendizado:** `lr=1e-2` foi a **única mudança que piorou** o resultado: F1 0,6631 e revocação 0,5955
  (−0,021 de revocação em relação à base). O passo 10× maior deixa a otimização mais instável e o modelo fica mais
  "conservador" (precisão maior, 0,7489, à custa da revocação).
- O desvio padrão da revocação do MLP (0,015–0,026) é maior que o do KNN e do SVM (~0,005–0,010): o MLP é mais
  sensível ao fold e ao ponto em que o early stopping para.

**Melhor configuração: MLP (128,64) relu** (pelo F1).

## A7. Exportação dos resultados

In [ ]:
tab = pd.DataFrame(resultados)
tab.to_csv('resultados/resultados.csv', index=False)
with open('resultados/matrizes.json', 'w') as f:
    # formato: [[TN, FP], [FN, TP]]  (linhas = real, colunas = previsto; 0 = <=50K, 1 = >50K)
    json.dump({k: v.tolist() for k, v in matrizes.items()}, f, indent=1)

pd.set_option('display.width', 200)
print(tab[['modelo', 'acc_media', 'acc_dp', 'prec_media', 'rec_media', 'f1_media', 'f1_dp', 'tempo_s']].round(4))

## B5. Consolidação dos resultados (Pessoa 2)
Lê os arquivos exportados em A7 (KNN, SVM e MLP juntos) e gera, em 300 dpi:
1. a **tabela final** (média ± desvio padrão dos 5 folds) → `Pessoa 2/resultados/tabela_final.csv`;
2. as **matrizes de confusão** do melhor modelo de cada família → `Pessoa 2/figuras/matrizes_confusao.png` (3 lado a lado)
   e `Pessoa 2/figuras/cm_<modelo>.png` (individuais);
3. o **gráfico de barras** de acurácia e F1 → `Pessoa 2/figuras/comparacao_modelos.png`.

Critério de "melhor modelo": **F1 da classe `>50K`**, mais adequado que a acurácia por causa do desbalanceamento.

In [ ]:
res = pd.read_csv('resultados/resultados.csv')
with open('resultados/matrizes.json') as f:
    mats = {k: np.array(v) for k, v in json.load(f).items()}
res['familia'] = res['modelo'].str.split().str[0]

def pm(media, dp):   # "0,8577 ± 0,0037" (vírgula decimal, para o artigo em português)
    return f"{media:.4f} ± {dp:.4f}".replace('.', ',')

tabela = pd.DataFrame({
    'Modelo':    res['modelo'],
    'Acurácia':  [pm(a, b) for a, b in zip(res['acc_media'],  res['acc_dp'])],
    'Precisão':  [pm(a, b) for a, b in zip(res['prec_media'], res['prec_dp'])],
    'Revocação': [pm(a, b) for a, b in zip(res['rec_media'],  res['rec_dp'])],
    'F1':        [pm(a, b) for a, b in zip(res['f1_media'],   res['f1_dp'])],
    'Tempo (s)': res['tempo_s'].map(lambda t: f"{t:.1f}".replace('.', ',')),
})
tabela.to_csv('Pessoa 2/resultados/tabela_final.csv', index=False, encoding='utf-8-sig')

melhores = (res.loc[res.groupby('familia')['f1_media'].idxmax()]
               .set_index('familia')['modelo'].reindex(['KNN', 'SVM', 'MLP']).dropna())
baseline = 1 - y.mean()   # acurácia de quem sempre prevê <=50K
print('Melhor de cada família (F1):', dict(melhores))
print(f'Acurácia do classificador trivial (sempre <=50K): {baseline:.4f}')
print('Melhor acurácia:', res.loc[res['acc_media'].idxmax(), 'modelo'],
      '| melhor F1:', res.loc[res['f1_media'].idxmax(), 'modelo'])
tabela

In [ ]:
import re
from sklearn.metrics import ConfusionMatrixDisplay
rot = ['<=50K', '>50K']
br = lambda n: f"{n:,}".replace(',', '.')

def desenha_cm(ax, nome):
    cm = mats[nome]
    pct = cm / cm.sum(axis=1, keepdims=True)          # % em relação à classe real (linha)
    pct_br = lambda p: f"{100 * p:.1f}".replace('.', ',')
    annot = [[f"{br(cm[i, j])}\n({pct_br(pct[i, j])}%)" for j in range(2)] for i in range(2)]
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False, ax=ax, square=True,
                xticklabels=rot, yticklabels=rot, linewidths=2, linecolor='white',
                annot_kws={'size': 11})
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real'); ax.set_title(nome, fontsize=11)

# Figura com as 3 matrizes lado a lado (economiza espaço no artigo)
fig, axs = plt.subplots(1, len(melhores), figsize=(4.2 * len(melhores), 3.9))
for ax, nome in zip(np.atleast_1d(axs), melhores):
    desenha_cm(ax, nome)
plt.tight_layout(); plt.savefig('Pessoa 2/figuras/matrizes_confusao.png', dpi=300); plt.show()

# Matrizes individuais (formato do guia)
for nome in melhores:
    ConfusionMatrixDisplay(mats[nome], display_labels=rot).plot(cmap='Blues', values_format='d', colorbar=False)
    arq = re.sub(r'[^0-9A-Za-z=]+', '_', nome).strip('_')
    plt.title(nome); plt.tight_layout(); plt.savefig(f'Pessoa 2/figuras/cm_{arq}.png', dpi=300); plt.close()

for nome in melhores:
    (tn, fp), (fn, tp) = mats[nome]
    print(f"{nome:22s} FP={br(fp)}  FN={br(fn)}  -> {fn / (tp + fn):.1%} dos '>50K' classificados como '<=50K'")

In [ ]:
# Gráfico de barras: acurácia e F1 (média ± desvio padrão dos 5 folds) de todas as configurações
ordem = res.iloc[::-1]          # primeiro modelo no topo
pos = np.arange(len(ordem)); h = 0.38
fig, ax = plt.subplots(figsize=(8, 0.42 * len(ordem) + 1.4))
for desl, col, rotulo, cor in [(h / 2, 'acc', 'Acurácia', '#2a78d6'), (-h / 2, 'f1', 'F1 (classe >50K)', '#eb6834')]:
    b = ax.barh(pos + desl, ordem[f'{col}_media'], height=h - 0.04, color=cor, label=rotulo,
                xerr=ordem[f'{col}_dp'], error_kw={'elinewidth': 1, 'capsize': 2, 'ecolor': '#555'})
    ax.bar_label(b, labels=[f"{v:.3f}".replace('.', ',') for v in ordem[f'{col}_media']],
                 padding=6, fontsize=7.5, color='#333')
ax.axvline(baseline, color='#555', ls='--', lw=1)
ax.text(baseline, -0.9, f' trivial (sempre <=50K) = {baseline:.3f}'.replace('.', ','),
        fontsize=8, color='#555', va='center')
ax.set_yticks(pos, ordem['modelo']); ax.set_xlim(0, 1); ax.set_ylim(-1.4, len(ordem) - 0.5)
ax.set_xlabel('Média dos 5 folds (barra de erro = desvio padrão)')
ax.set_title('Acurácia e F1 por configuração', pad=28)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', alpha=.3)
ax.xaxis.set_major_formatter(lambda v, _: f'{v:.1f}'.replace('.', ','))
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig('Pessoa 2/figuras/comparacao_modelos.png', dpi=300); plt.show()

### Resumo dos achados (Pessoa 2)
1. **Melhor acurácia:** SVM RBF C=1 (0,8577 ± 0,0037), praticamente empatado com MLP (128,64) (0,8570) e
   SVM RBF C=10 (0,8568). As diferenças são menores que o desvio padrão.
2. **Melhor F1 na classe `>50K`:** **MLP (128,64) relu (0,6762 ± 0,0122)**. 5 das 6 configurações do MLP
   (F1 0,672–0,676) superaram o melhor SVM (0,6687) e o melhor KNN (0,6587), principalmente por terem
   **revocação maior** (~0,62 contra ~0,60); só a de `lr=1e-2` (0,6631) ficou abaixo do SVM.
3. **A acurácia é enganosa:** prever sempre `<=50K` já dá 0,7606. Os melhores modelos ficam só ~10 pontos acima,
   e o F1 da classe de interesse não passa de ~0,68.
4. **MLP mais profundo não compensou:** 3 camadas não melhoraram nada (F1 0,6718), levaram ~1,9× o tempo da
   base e mostraram sinal de overfitting. Só aumentar a taxa de aprendizado piorou o resultado.
5. **Precisão > revocação em todos os modelos:** os erros são dominados por **falsos negativos**. O melhor MLP
   tem 4.388 FN contra 2.591 FP, ou seja, 37,6% das pessoas de renda `>50K` são classificadas como `<=50K`
   (39,6% no SVM RBF C=10 e 39,2% no KNN k=31). Isso é efeito do desbalanceamento 76/24.
6. **Custo:** o SVM RBF levou 314–396 s (5 folds), o MLP 11–26 s e o KNN ~4 s. O melhor MLP (21 s) teve F1 maior que o
   melhor SVM (396 s) gastando ~19 vezes menos tempo.

### Download dos arquivos (Colab)

In [ ]:
# No Colab: baixa figuras e resultados num .zip
try:
    from google.colab import files
    import shutil
    shutil.make_archive('saida_grupo6', 'zip', '.', 'figuras')
    shutil.make_archive('resultados_grupo6', 'zip', '.', 'resultados')
    shutil.make_archive('pessoa2_grupo6', 'zip', '.', 'Pessoa 2')
    files.download('saida_grupo6.zip'); files.download('resultados_grupo6.zip'); files.download('pessoa2_grupo6.zip')
except ImportError:
    print('Fora do Colab: arquivos em ./figuras e ./resultados')